### (1) Import Libraries and Processed Data
- `sklearn` for model building, evaluation, and hyperparameter tuning
- `xgboost` for gradient boosting models
- `shap` for explaining feature importance
- `joblib` to save trained models
- `os` to create directories for saving models

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

import sys
!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install shap

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import shap
# import joblib
# import os

sns.set_style("whitegrid")

# Import Processed Data
X_train = pd.read_csv("../data/processed_data/X_train.csv")
X_test = pd.read_csv("../data/processed_data/X_test.csv")
y_train = pd.read_csv("../data/processed_data/y_train.csv")
y_test = pd.read_csv("../data/processed_data/y_test.csv")

target_col = "churned_30d"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [shap]


### (2) Train a baseline model


In [5]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   user_id              80000 non-null  object
 1   signup_date          80000 non-null  object
 2   plan_tier            80000 non-null  object
 3   company_size         80000 non-null  object
 4   industry             80000 non-null  object
 5   acquisition_channel  80000 non-null  object
 6   is_enterprise        80000 non-null  bool  
 7   downgraded           80000 non-null  int64 
 8   expansion_event      80000 non-null  int64 
 9   region_imputed       80000 non-null  object
dtypes: bool(1), int64(2), object(7)
memory usage: 5.6+ MB


In [ ]:
# Random Forest Baseline Model

# Creates a collection of decision trees - RF trains many trees on many subsets and combines their predictions
rf_model = RandomForestClassifier(
    random_state=42
)

# should fail here > original X_train contains one or more columns with raw text > RF in sklearn can't process strings
rf_model.fit(X_train, y_train.values.ravel())

# Make predictions
y_pred = rf_model.predict(X_test)

# Predict Probabilities
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

ValueError: could not convert string to float: '8feaab18-4e4b-4280-90cb-c5a2cc2c38a4'

### (3) Evaluate Model

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
# F1 balances precision and recall
print("F1:", f1_score(y_test, y_pred))
# AUC checks how well results are distinguished from each other
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))

# Confusion Matrix - ideally large TN, TP and small FP, FN - consider which is more costly
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

### (4) Hyperparameter Tuning

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

# Randomised Search - tries a range of random forest settings (e.g. underfit vs overfit) and keeps the best
# larger values = simpler trees, prevents overfitting
rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train.values.ravel())

print("Best Parameters:")
print(rf_search.best_params_)

### (5) Evaluate tuned model

In [ ]:
best_rf = rf_search.best_estimator_

y_pred = best_rf.predict(X_test)
y_pred_proba = best_rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

# Check performance of tuned vs un-tuned model
print(
    "ROC AUC:",
    roc_auc_score(y_test, y_pred_proba)
)

# Feature Importance - which features helped the most when making decisions
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print(importance_df.head(10))

# Visual Feature importance_df
plt.figure(figsize=(8,6))

sns.barplot(
    data=importance_df.head(10),
    x="importance",
    y="feature"
)

plt.title("Top 10 Important Features")
plt.show()